#Распаковка архива

In [ ]:
!unzip /content/foto1.zip -d /content

#Импорт библиотек/Настройка параметров

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import re
from scipy.ndimage import binary_fill_holes
import cv2
import gc
import shutil
from skimage.morphology import disk
from skimage.filters import median

In [ ]:
# Параметры
IMG_DIR = '/content/foto1'  # Путь к изображениям
THRESHOLD_BRIGHT = 1        # Порог яркости для фона
THRESHOLD_STD = 50          # Порог стабильности
OUTPUT_DIR = '/content/'
BATCH_SIZE = 200            # Оптимальный размер батча для 12ГБ ОЗУ

In [ ]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def process_images_to_max_pixels(img_dir, output_path):
    # Получаем список изображений с поддержкой естественной сортировки
    files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.png', '.bmp'))], key=natural_sort_key)

    if not files:
        print("Нет изображений для обработки")
        return

    # Инициализируем массив для максимальных значений
    max_pixels = None

    for i, filename in enumerate(tqdm(files, desc="Обработка изображений")):
        filepath = os.path.join(img_dir, filename)
        img = cv2.imread(filepath)

        if img is None:
            tqdm.write(f"Не удалось загрузить изображение: {filename}")  # Используем tqdm.write для корректного вывода
            continue

        if max_pixels is None:
            # Инициализируем массив при первой итерации
            max_pixels = img.copy().astype(np.float32)
        else:
            if img.shape != max_pixels.shape:
                tqdm.write(f"Изображение {filename} имеет другой размер и будет пропущено")
                continue

            # Находим максимумы для каждого пикселя
            max_pixels = np.maximum(max_pixels, img.astype(np.float32))

    if max_pixels is not None:
        # Конвертируем обратно в uint8 и сохраняем
        max_pixels = np.clip(max_pixels, 0, 255).astype(np.uint8)
        cv2.imwrite(output_path, max_pixels)
        print(f"Результирующее изображение сохранено как {output_path}")

        plt.figure(figsize=(10, 10))
        # Конвертируем из BGR (OpenCV) в RGB (matplotlib)
        plt.imshow(cv2.cvtColor(max_pixels, cv2.COLOR_BGR2RGB))
        plt.axis('off')  # Скрываем оси
        plt.show()

    else:
        print("Не удалось создать результирующее изображение")

# Пример использования
OUTPUT_IMAGE = "max_original_image.png"
process_images_to_max_pixels(IMG_DIR, OUTPUT_IMAGE)

#Обрезка изображений

In [ ]:
def interactive_crop(image_path):
    """Интерактивная обрезка, всегда применяемая к исходному изображению"""
    plt.ion()

    img = Image.open(image_path)
    original_array = np.array(img)  # Сохраняем оригинал
    current_array = original_array.copy()  # Текущее состояние
    crop_history = []

    while True:
        # Создаем фигуру
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # Панель 1: Оригинал с границами
        ax1.imshow(original_array, cmap='gray')
        ax1.set_title("Исходное изображение с границами")

        # Панель 2: Предыдущий результат
        ax2.imshow(current_array, cmap='gray')
        ax2.set_title("Текущий результат обрезки")

        # Рисуем границы на оригинале
        if crop_history:
            last_left, last_right, last_bottom, last_top = crop_history[-1]
            h, w = original_array.shape[:2]

            # Прямоугольник обрезки
            rect = plt.Rectangle(
                (last_left, last_top),
                w - last_left - last_right,
                h - last_top - last_bottom,
                fill=False, edgecolor='red', linewidth=2, linestyle='--'
            )
            ax1.add_patch(rect)

            # Текст с параметрами
            info_text = f"Последние параметры:\nleft: {last_left}\nright: {last_right}\n"
            info_text += f"top: {last_top}\nbottom: {last_bottom}"
            ax1.text(10, 30, info_text, color='yellow',
                    bbox=dict(facecolor='black', alpha=0.7))

        plt.tight_layout()
        plt.show(block=False)
        plt.pause(0.1)

        # Запрос параметров
        print("\n" + "="*50)
        print(f"Исходный размер: {original_array.shape[1]}x{original_array.shape[0]}")
        print("Введите отступы в пикселях(Enter для предыдущего результата):")

        last_crop = crop_history[-1] if crop_history else (0, 0, 0, 0)
        try:
            left = int(input(f"Слева [{last_crop[0]}]: ") or last_crop[0])
            right = int(input(f"Справа [{last_crop[1]}]: ") or last_crop[1])
            bottom = int(input(f"Снизу [{last_crop[2]}]: ") or last_crop[2])
            top = int(input(f"Сверху [{last_crop[3]}]: ") or last_crop[3])
        except ValueError:
            print("Ошибка! Вводите только целые числа.")
            continue

        # Применяем обрезку к оригиналу
        h, w = original_array.shape[:2]
        try:
            current_array = original_array[top:h-bottom, left:w-right]
        except IndexError:
            print("Ошибка! Введенные значения выходят за границы.")
            continue

        crop_params = (left, right, bottom, top)
        crop_history.append(crop_params)

        # Закрываем предыдущие графики
        plt.close(fig)

        # Показываем результат
        plt.figure(figsize=(8, 8))
        plt.imshow(current_array, cmap='gray')
        plt.title(f"Результат обрезки: {current_array.shape[1]}x{current_array.shape[0]}")
        plt.show(block=False)
        plt.pause(0.1)

        confirm = input("Результат устраивает? (y/n): ").strip().lower()
        if confirm == 'y':
            plt.close('all')
            output_path = os.path.splitext(image_path)[0] + "_cropped.png"
            Image.fromarray(current_array).save(output_path)
            print(f"\nФинальные параметры: left={left}, right={right}, bottom={bottom}, top={top}")
            return output_path, crop_params
        else:
            plt.close('all')

def batch_crop(input_dir, output_dir, crop_params):
    """Пакетная обрезка (без изменений)"""
    os.makedirs(output_dir, exist_ok=True)
    left, right, bottom, top = crop_params

    files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg', '.bmp'))]
    print(f"Найдено {len(files)} изображений для обработки")

    for filename in tqdm(files, desc="Обрезка изображений"):
        try:
            img_path = os.path.join(input_dir, filename)
            img = Image.open(img_path)
            img_array = np.array(img)

            h, w = img_array.shape[:2]
            cropped = img_array[top:h-bottom, left:w-right]

            output_path = os.path.join(output_dir, f"cropped_{filename}")
            Image.fromarray(cropped).save(output_path)
        except Exception as e:
            print(f"\nОшибка при обработке {filename}: {str(e)}")

    print(f"\nГотово! Результаты сохранены в {output_dir}")

In [ ]:
# 1. Интерактивная обрезка
cropped_img, params = interactive_crop("max_original_image.png") # обрезаем на суммарном фото

# 2. Пакетная обработка
input_folder = IMG_DIR  # путь к фото, которые будем обрезать
small_folder = "/content/dataset/input" # путь, куда сохраняются обрезанные фото
batch_crop(input_folder, small_folder, params)

#Вычисление статического фона

In [ ]:
def compute_true_background(img_dir, batch_size=200, brightness_thresh=1, std_thresh=5):
    """Корректное вычисление статического фона"""
    files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.bmp'))], key=natural_sort_key)
    if not files:
        raise ValueError("No PNG files found in directory")

    # Получаем параметры первого изображения
    sample = np.array(Image.open(os.path.join(img_dir, files[0])).convert('L'))
    height, width = sample.shape

    # Инициализируем карты для суммы и счётчика
    sum_map = np.zeros((height, width), dtype=np.float64)
    count_map = np.zeros((height, width), dtype=np.int32)
    std_accumulator = np.zeros((height, width), dtype=np.float32)
    processed_batches = 0

    for i in tqdm(range(0, len(files), batch_size), desc="Processing batches"):
        batch_files = files[i:i + batch_size]

        # Загрузка батча
        batch = []
        for f in batch_files:
            try:
                img = np.array(Image.open(os.path.join(img_dir, f)).convert('L'))
                if img.shape == (height, width):
                    batch.append(img)
            except Exception as e:
                print(f"Error loading {f}: {str(e)}")
                continue

        if not batch:
            continue

        batch_array = np.stack(batch)
        batch_median = np.median(batch_array, axis=0)
        batch_std = np.std(batch_array.astype(np.float32), axis=0)

        # Маска стабильных пикселей
        stable_mask = (batch_std < std_thresh) & (batch_median > brightness_thresh)

        # Накопление суммы и счётчика для стабильных пикселей
        sum_map += np.where(stable_mask, batch_median, 0)
        count_map += stable_mask.astype(np.int32)

        # Аккумулируем std
        std_accumulator += batch_std
        processed_batches += 1

        # Очистка памяти
        del batch, batch_array, batch_median, batch_std, stable_mask
        gc.collect()

    if processed_batches == 0:
        raise ValueError("No valid batches processed")

    # Усредняем std карту
    std_map = std_accumulator / processed_batches

    # Вычисляем окончательный фон
    background = np.zeros((height, width), dtype=np.uint8)
    valid_mask = count_map > 0
    background[valid_mask] = (sum_map[valid_mask] / count_map[valid_mask]).astype(np.uint8)

    # Создаем окончательную маску фона
    background_mask = (std_map < std_thresh) & (background > brightness_thresh)
    final_background = np.where(background_mask, background, 0)

    return final_background.astype(np.uint8), std_map

In [ ]:
# Вычисление фона
background, std_map = compute_true_background(
        img_dir=small_folder,
        batch_size=200,
        brightness_thresh=THRESHOLD_BRIGHT,
        std_thresh=THRESHOLD_STD
    )

# Сохранение
Image.fromarray(background).save('/content/camera_background.png')
Image.fromarray((std_map * 255 / std_map.max()).astype(np.uint8)).save('/content/std_map.png')

# Визуализация
print('Вычисленный статический фон')
plt.figure(figsize=(10, 10))
# Конвертируем из BGR (OpenCV) в RGB (matplotlib)
plt.imshow(cv2.cvtColor(background, cv2.COLOR_BGR2RGB))
plt.axis('off')  # Скрываем оси
plt.show()

#Анализ вычитания статического фона

In [ ]:
# Функция вычитания фона
def subtract_background_batch(img_dir, background, background_mask, batch_size=200, output_dir=None):
    """
    Корректно вычитает фон из каждого изображения пакетами и сохраняет результаты.
    Поддерживает .png и .bmp файлы.
    """
    # Поддержка и .png, и .bmp файлов
    files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.bmp'))])

    if not files:
        raise ValueError("В указанной директории нет .png или .bmp файлов!")

    height, width = background.shape

    # Инициализация аккумуляторов статистик
    total_original = np.zeros((height, width), dtype=np.float64)
    total_cleaned = np.zeros((height, width), dtype=np.float64)
    max_original = np.zeros((height, width), dtype=np.uint8)
    max_without_background = np.zeros((height, width), dtype=np.float32)

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    for i in tqdm(range(0, len(files), batch_size), desc="Обработка батчей"):
        batch_files = files[i:i + batch_size]
        batch_cleaned = []

        for f in batch_files:
            img = np.array(Image.open(os.path.join(img_dir, f)))

            # Проверка на соответствие размеров
            if img.shape != background.shape:
                print(f"Предупреждение: изображение {f} имеет размер {img.shape}, ожидается {background.shape}. Пропускаем.")
                continue

            total_original += img

            cleaned = img.astype(np.float32)
            cleaned[background_mask] -= background[background_mask]
            cleaned = np.clip(cleaned, 0, 255).astype(np.uint8)  # Ограничение значений и конвертация

            batch_cleaned.append(cleaned)

        if not batch_cleaned:  # Пропускаем пустые батчи
            continue

        batch_cleaned = np.array(batch_cleaned)
        total_cleaned += np.sum(batch_cleaned, axis=0)

        # Максимумы только для успешно обработанных изображений
        max_original = np.maximum(max_original, np.max([np.array(Image.open(os.path.join(img_dir, f))) for f in batch_files if os.path.exists(os.path.join(img_dir, f))], axis=0))
        max_without_background = np.maximum(max_without_background, np.max(batch_cleaned, axis=0))

    # Нормализация только если есть файлы
    if len(files) > 0:
        total_original /= len(files)
        total_cleaned /= len(files)
    else:
        print("Предупреждение: Нет файлов для обработки!")
        return None

    return {
        'total_original': total_original.astype(np.float32),
        'total_cleaned': total_cleaned.astype(np.float32),
        'max_original': max_original.astype(np.uint8),
        'max_without_background': max_without_background.astype(np.uint8),
        'cleaned_images': output_dir if output_dir else None
    }

In [ ]:
# Загрузка вычисленного фона
background = np.array(Image.open('/content/camera_background.png'))
background_mask = background > 0  # Маска фона

# Обработка с пакетированием
results = subtract_background_batch(
    img_dir=small_folder,
    background=background,
    background_mask=background_mask,
    batch_size=200,
    output_dir=OUTPUT_DIR
)

# Сохранение результатов
output_path = OUTPUT_DIR
Image.fromarray(results['max_without_background']).save(os.path.join(output_path, "max_without_background.bmp"))

# Визуализация (аналогично вашему коду)
plt.figure(figsize=(24, 12))

plt.subplot(1, 2, 1)
plt.imshow(results['max_without_background'], cmap='gray', vmin=0, vmax=255)
plt.title('Максимумы (очищенные)')

plt.subplot(1, 2, 2)
intensity_orig = np.sum(results['total_original'], axis=0)
intensity_clean = np.sum(results['total_cleaned'], axis=0)
plt.plot(intensity_orig / intensity_orig.max(), label="Исходные", color='blue')
plt.plot(intensity_clean / intensity_clean.max(), label="Без фона", color='orange')
plt.title("Горизонтальная интенсивность")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

#Функции очистки



>     Из каждого изображения вычитается найденный ранее статичный фон.

    
>     Если пиксель изолирован и вокруг него средняя яркость меньше его самого — отбрасывается как шум.





In [ ]:
# Функция для вычисления интенсивности пикселей
def calculate_intensity(img):
    return np.sum(img, axis=0)


#  Оптимизированная версия удаления горячих пикселей
def fast_remove_hot_pixels(image, intensity_threshold=50, neighbor_threshold=5, radius=2):
    cleaned = image.copy()
    height, width = image.shape
    padded = np.pad(image, radius, mode='constant')

    for y in range(radius, height + radius):
        for x in range(radius, width + radius):
            val = padded[y, x]
            if val > intensity_threshold:
                neighborhood = padded[y-radius:y+radius+1, x-radius:x+radius+1]
                neighborhood[radius, radius] = 0
                mean_neighbors = np.mean(neighborhood[neighborhood > 0]) if np.any(neighborhood > 0) else 0
                if mean_neighbors < neighbor_threshold and val > 3 * mean_neighbors:
                    cleaned[y-radius, x-radius] = 0
    return cleaned


def universal_pixel_cleaner(
    image,
    hot_pixel_params={
        'intensity_threshold': 50,
        'neighbor_threshold': 5,
        'radius': 2
    },
    cluster_params={
        'intensity_ratio': 4.0,
        'similarity_threshold': 0.15,
        'cluster_radius': 1
    },
    debug=False
):
    """
    Универсальный очиститель битых пикселей и их кластеров.
    Теперь без сложных векторизаций и с защитой от всех ошибок.
    """
    try:
        # Конвертируем в одноканальное изображение
        if len(image.shape) == 3:
            if image.shape[2] in (3, 4):  # RGB/RGBA
                image = np.mean(image[..., :3], axis=2).astype(np.uint8)

        img_float = image.astype(np.float32)
        cleaned = image.copy()
        h, w = image.shape

        # --- Шаг 1: Удаление одиночных горячих пикселей ---
        padded = np.pad(image, hot_pixel_params['radius'], mode='reflect')
        radius = hot_pixel_params['radius']
        for y in range(h):
            for x in range(w):
                pixel_val = image[y, x]
                if pixel_val <= hot_pixel_params['intensity_threshold']:
                    continue

                # Берём окрестность
                y_start = y + radius - radius
                y_end = y + radius + radius + 1
                x_start = x + radius - radius
                x_end = x + radius + radius + 1
                neighborhood = padded[y_start:y_end, x_start:x_end]

                # Убираем центр
                cy, cx = radius, radius
                neighborhood[cy, cx] = 0

                mean_neighbors = np.mean(neighborhood[neighborhood > 0]) if np.any(neighborhood > 0) else 0
                if mean_neighbors < hot_pixel_params['neighbor_threshold'] and pixel_val > 3 * mean_neighbors:
                    cleaned[y, x] = 0

        # --- Шаг 2: Удаление кластеров аномальных пикселей ---
        if cluster_params:
            window_size = 2 * cluster_params['cluster_radius'] + 1
            avg_kernel = np.ones((window_size, window_size), dtype=np.float32) / (window_size ** 2)
            local_avg = cv2.filter2D(img_float, -1, avg_kernel)

            # Защита от деления на 0
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio = img_float / (local_avg + 1e-6)

            is_anomalous = (ratio > cluster_params['intensity_ratio']) | (ratio < 1.0 / cluster_params['intensity_ratio'])

            # Простое сглаживание для проверки однородности
            smoothed = cv2.GaussianBlur(img_float, (5, 5), 1.0)
            diff_to_smooth = np.abs(img_float - smoothed)
            is_uniform = diff_to_smooth < cluster_params['similarity_threshold'] * 255

            # Комбинируем условия
            cluster_mask = is_anomalous & is_uniform
            cleaned[cluster_mask] = 0

        # --- Шаг 3: Медианный фильтр ---
        cleaned = median(cleaned, disk(1))

        return cleaned.astype(np.uint8)

    except Exception as e:
        print(f"[Ошибка при обработке изображения]: {e}")
        return image.astype(np.uint8)

#Функции создания датасета и анализа результатов очистки

In [ ]:
# Функция для архивации датасета
def zip_dataset(output_dir="dataset", zip_filename="dataset.zip"):
    """
    Архивирует датасет в ZIP-файл.

    :param output_dir: Директория с датасетом.
    :param zip_filename: Имя ZIP-файла.
    """
    # Создание ZIP-архива
    shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', output_dir)
    print(f"Датасет заархивирован в файл: {zip_filename}")

def analyze_dataset(input_dir, bg_sub_dir, target_dir, output_path="analysis_results"):
    """
    Анализирует интенсивность по трем этапам обработки и строит столбчатую диаграмму.
    Также находит максимальные значения пикселей среди всех очищенных изображений.
    """
    os.makedirs(output_path, exist_ok=True)

    def load_and_sum(dir_path):
        """Загружает изображения и суммирует интенсивность"""
        files = sorted([f for f in os.listdir(dir_path) if f.endswith(('.png', '.bmp'))])
        total_intensity = 0
        for f in files:
            img = np.array(Image.open(os.path.join(dir_path, f)))
            total_intensity += np.sum(img)
        return total_intensity

    # --- Расчёт суммарной интенсивности ---
    input_sum = load_and_sum(input_dir)
    bg_sub_sum = load_and_sum(bg_sub_dir)
    target_sum = load_and_sum(target_dir)

    # Нормировка к максимальному значению (для графика)
    max_sum = max(input_sum, bg_sub_sum, target_sum)
    if max_sum > 0:
        input_norm = input_sum / max_sum
        bg_sub_norm = bg_sub_sum / max_sum
        target_norm = target_sum / max_sum
    else:
        input_norm, bg_sub_norm, target_norm = 0, 0, 0

    # --- Построение графика ---
    labels = ['Исходные', 'После фона', 'После очистки']
    values = [input_norm, bg_sub_norm, target_norm]
    raw_values = [input_sum, bg_sub_sum, target_sum]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(labels, values, color=['blue', 'orange', 'green'], alpha=0.7)

    for bar, raw_val in zip(bars, raw_values):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                 f'{raw_val:.1f}\n({height:.1%})',
                 ha='center', va='bottom')

    plt.title('Сравнение суммарной интенсивности по этапам обработки')
    plt.ylabel('Нормированная интенсивность')
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(output_path, 'intensity_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print("Результаты анализа:")
    print(f"1. Исходные: {input_sum:.1f} ({input_norm:.1%})")
    print(f"2. После вычитания фона: {bg_sub_sum:.1f} ({bg_sub_norm:.1%})")
    print(f"3. После очистки: {target_sum:.1f} ({target_norm:.1%})")
    print(f"Графики сохранены в: {output_path}")

    # --- Нахождение максимума по пикселям только для очищенных изображений ---
    print("\nАнализ максимумов по пикселям для очищенных изображений...")

    target_files = sorted([f for f in os.listdir(target_dir) if f.endswith(('.png', '.bmp'))])

    if not target_files:
        print("❌ Очищенные изображения не найдены.")
        return

    # Загрузка первого изображения для инициализации массива
    first_img = np.array(Image.open(os.path.join(target_dir, target_files[0])))
    max_pixel_image = first_img.copy().astype(np.float32)

    # Поиск максимума по каждому пикселю
    for filename in target_files[1:]:
        img = np.array(Image.open(os.path.join(target_dir, filename)))
        max_pixel_image = np.maximum(max_pixel_image, img)

    # Сохранение результата
    result_path = os.path.join(output_path, "max_cleaned_image.png")
    Image.fromarray(max_pixel_image.astype(np.uint8)).save(result_path)

    print(f"\n✅ Максимумы по пикселям рассчитаны и сохранены в: {result_path}")
    print(f"Минимальное значение на карте максимумов: {np.min(max_pixel_image)}")
    print(f"Максимальное значение на карте максимумов: {np.max(max_pixel_image)}")

    # Отображение изображения (если разрешено)
    try:
        plt.figure(figsize=(10, 8))
        plt.imshow(max_pixel_image, cmap='gray')
        plt.title('Карта максимальных значений пикселей (очищенные изображения)')
        plt.axis('off')
        plt.show()
    except:
        print("⚠️ Не удалось отобразить изображение (возможно, в консоли без GUI).")


def compare_cleaning_stages(cleaned_dir, final_dir, output_path="cleaning_comparison"):
    """
    Сравнивает интенсивности после вычитания фона и после финальной очистки

    Параметры:
        cleaned_dir: Папка с изображениями после вычитания фона
        final_dir: Папка с финальными обработанными изображениями
        output_path: Папка для сохранения результатов
    """
    os.makedirs(output_path, exist_ok=True)

    def load_and_sum(dir_path):
        """Загружает изображения и суммирует интенсивность по горизонтали"""
        files = sorted([f for f in os.listdir(dir_path) if f.endswith(('.png', '.bmp'))])
        total_intensity = None
        for f in files:
            img = np.array(Image.open(os.path.join(dir_path, f)))
            # Суммируем по вертикали (горизонтальная интенсивность)
            img_sum = np.sum(img, axis=0)
            if total_intensity is None:
                total_intensity = img_sum
            else:
                total_intensity += img_sum
        return total_intensity

    # Получаем горизонтальные интенсивности
    cleaned_intensity = load_and_sum(cleaned_dir)
    final_intensity = load_and_sum(final_dir)

    # Нормировка к максимальному значению
    max_intensity = max(np.max(cleaned_intensity), np.max(final_intensity))
    if max_intensity > 0:
        cleaned_norm = cleaned_intensity / max_intensity
        final_norm = final_intensity / max_intensity
    else:
        cleaned_norm = np.zeros_like(cleaned_intensity)
        final_norm = np.zeros_like(final_intensity)

    # Создаем график
    plt.figure(figsize=(16, 10))

    plt.plot(cleaned_norm, label="После вычитания фона", color='orange', linewidth=2)
    plt.plot(final_norm, label="После финальной очистки", color='blue', linewidth=1, linestyle='--')

    plt.title("Сравнение горизонтальной интенсивности после этапов обработки")
    plt.xlabel("Пиксель X")
    plt.ylabel("Нормированная интенсивность")
    plt.legend()
    plt.grid(True)

    # Сохранение результатов
    plot_path = os.path.join(output_path, 'cleaning_comparison_plot.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"График сохранен в {plot_path}")

In [ ]:
def create_fast_dataset(imgs_array, background, background_mask, camera_id, output_dir="dataset",
                      hot_pixel_params={'intensity_threshold':50, 'neighbor_threshold':5, 'radius':2}, cluster_params={'intensity_ratio': 4.0, 'similarity_threshold': 0.15,'cluster_radius': 1}):
    """
    Ускоренная версия с предзагруженными данными в памяти.
    Обрабатывает 300 изображений 1360x1040 за ~2-3 минуты.
    """
    # Создаем директории
    input_dir = os.path.join(output_dir, "input")
    bg_sub_dir = os.path.join(output_dir, "bg_subtracted")
    target_dir = os.path.join(output_dir, "target")

    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(bg_sub_dir, exist_ok=True)
    os.makedirs(target_dir, exist_ok=True)

    # Предварительные вычисления
    bg_correction = np.where(background_mask, background, 0)

    for i in tqdm(range(len(imgs_array)), desc="Обработка"):
        # 1. Сохраняем оригинал
        Image.fromarray(imgs_array[i]).save(os.path.join(input_dir, f"{camera_id}_image_{i+1}.png"))

        # 2. Вычитаем фон (векторизованная операция)
        bg_subtracted = np.clip(imgs_array[i].astype(np.float32) - bg_correction, 0, 255).astype(np.uint8)
        Image.fromarray(bg_subtracted).save(os.path.join(bg_sub_dir, f"{camera_id}_image_{i+1}.png"))

        # 3. Оптимизированное удаление горячих пикселей
        final_image = universal_pixel_cleaner(bg_subtracted, hot_pixel_params=hot_pixel_params, cluster_params=cluster_params)
        Image.fromarray(final_image).save(os.path.join(target_dir, f"{camera_id}_image_{i+1}.png"))

    print(f"Датасет создан за {len(imgs_array)} изображений")
    return {'input_dir': input_dir, 'bg_sub_dir': bg_sub_dir, 'target_dir': target_dir}

#Создание датасета

In [ ]:
# Получаем список изображений с естественной сортировкой
files = sorted([f for f in os.listdir(small_folder) if f.lower().endswith(('.png', '.bmp', '.jpg', '.jpeg'))],
               key=natural_sort_key)

if not files:
    raise ValueError(f"В папке {small_folder} не найдено подходящих изображений (.png, .bmp, .jpg, .jpeg)")


# Загрузка всех изображений с проверкой размеров
images = []
base_shape = None

for file in tqdm(files, desc="Загрузка изображений"):
    img_path = os.path.join(small_folder, file)
    try:
        img = np.array(Image.open(img_path))

        # Проверяем, что изображение успешно загружено
        if img.size == 0:
            print(f"Предупреждение: {file} не удалось загрузить (размер 0), пропускаем")
            continue

        # Проверяем одинаковость размеров всех изображений
        if base_shape is None:
            base_shape = img.shape
        elif img.shape != base_shape:
            print(f"Предупреждение: {file} имеет размер {img.shape}, ожидается {base_shape}, пропускаем")
            continue

        images.append(img)
    except Exception as e:
        print(f"Ошибка при загрузке {file}: {str(e)}")
        continue

if not images:
    raise ValueError("Не удалось загрузить ни одного изображения")

try:
    # Создаем массив из всех изображений
    imgs_array = np.stack(images)
    print(f"Успешно создан массив изображений размерностью {imgs_array.shape}")
except ValueError as e:
    raise ValueError(f"Не удалось создать массив изображений: {str(e)}. Возможно, изображения имеют разные размеры.")

In [ ]:
# Создание датасета
create_fast_dataset(imgs_array, background, background_mask, camera_id="cam1", output_dir="dataset",
                      hot_pixel_params={'intensity_threshold':1, 'neighbor_threshold':1, 'radius':1}, cluster_params={'intensity_ratio': 4.0,'similarity_threshold': 0.15, 'cluster_radius': 1})

In [ ]:
# Архивация датасета
zip_dataset(output_dir="dataset", zip_filename="dataset.zip")

#Анализ методов очистки

In [ ]:
analyze_dataset(
    input_dir="dataset/input",
    bg_sub_dir="dataset/bg_subtracted",
    target_dir="dataset/target",
    output_path=OUTPUT_DIR
)

In [ ]:
compare_cleaning_stages(
    cleaned_dir="dataset/bg_subtracted", # после вычитания фона
    final_dir="dataset/target",          # финальная очистка
    output_path="content" # сохранение фото с графиком
)